In [ ]:
# download the weaviate client
%pip install -U weaviate-client

In [ ]:
import weaviate, os
from weaviate.config import AdditionalConfig, Timeout
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv(override=True)

# Retrieve environment variables
CLUSTER_URL = os.getenv("CLUSTER_URL")
API_KEY = os.getenv("API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
COHERE_API_KEY = os.getenv("COHERE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# Connect to Weaviate
client = weaviate.connect_to_weaviate_cloud(
	cluster_url=CLUSTER_URL,
	auth_credentials=weaviate.auth.AuthApiKey(API_KEY),
	headers={
		"X-OpenAI-Api-Key": OPENAI_API_KEY,
		"X-Cohere-Api-Key": COHERE_API_KEY,
        "X-Goog-Api-Key": GOOGLE_API_KEY
	},
	additional_config=AdditionalConfig(
		timeout=Timeout(init=30, query=60, insert=120)
	)
)

ready = client.is_ready()
server_version = client.get_meta()["version"]
client_version = weaviate.__version__
live = client.is_live()
connected = client.is_connected()

print(f"Weaviate Ready: {ready}")
print(f"Weaviate Client Version: {client_version}")
print(f"Weaviate Server Version: {server_version}")
print(f"Weaviate Live: {client.is_live()}")
print(f"Client Connected: {connected}")

In [ ]:
import json

# Get cluster statistics
stat = client.cluster.statistics()
# Format and print cluster statistics in an organized way
print("=" * 80)
print("CLUSTER STATISTICS")
print("=" * 80)
print(f"\nSynchronized: {stat.synchronized}\n")

for i, node in enumerate(stat.statistics, 1):
    print(f"\n{'─' * 80}")
    print(f"NODE {i}: {node.name}")
    print(f"{'─' * 80}")
    print(f"Status: {node.status}")
    print(f"Ready: {node.ready}")
    print(f"DB Loaded: {node.db_loaded}")
    print(f"Is Open: {node.is_open}")
    print(f"Is Voter: {node.is_voter}")
    print(f"Leader ID: {node.leader_id}")
    print(f"Leader Address: {node.leader_address}")
    print(f"Initial Last Applied Index: {node.initial_last_applied_index}")
    
    print(f"\nRAFT STATS:")
    print(f"  State: {node.raft.state}")
    print(f"  Term: {node.raft.term}")
    print(f"  Applied Index: {node.raft.applied_index}")
    print(f"  Commit Index: {node.raft.commit_index}")
    print(f"  Last Log Index: {node.raft.last_log_index}")
    print(f"  Last Log Term: {node.raft.last_log_term}")
    print(f"  Last Snapshot Index: {node.raft.last_snapshot_index}")
    print(f"  Last Snapshot Term: {node.raft.last_snapshot_term}")
    print(f"  Last Contact: {node.raft.last_contact}")
    print(f"  FSM Pending: {node.raft.fsm_pending}")
    print(f"  Num Peers: {node.raft.num_peers}")
    print(f"  Protocol Version: {node.raft.protocol_version}")
    print(f"  Protocol Version Min: {node.raft.protocol_version_min}")
    print(f"  Protocol Version Max: {node.raft.protocol_version_max}")
    print(f"  Snapshot Version Min: {node.raft.snapshot_version_min}")
    print(f"  Snapshot Version Max: {node.raft.snapshot_version_max}")
    
    if node.raft.latest_configuration:
        print(f"\n  RAFT CONFIGURATION:")
        for member in node.raft.latest_configuration:
            print(f"    - Node ID: {member.node_id}, Address: {member.address}, Suffrage: {member.suffrage}")

print(f"\n{'=' * 80}\n")

In [ ]:
# Get the meta information of the Weaviate instance
result = client.get_meta()
print(result)

In [ ]:
# Read Shards status
coll = client.collections.use("<COLLECTION_NAME>")
shards = coll.shards()
print(shards)

In [ ]:
# Get the node and shard information into a table
from prettytable import PrettyTable

node_info = client.cluster.nodes(output="verbose") #'minimal' info, verbose for more details
print(node_info)

node_info = client.cluster.nodes(collection="<collection_name>", output="verbose")
print(node_info)

# shard_table = PrettyTable()
# shard_table.field_names = ["Node Name", "Collection", "Shard Name", "Object Count", "Index Status", "Loaded"]
# for node in node_info:
#     for shard in node.shards:
#         shard_table.add_row([node.name, shard.collection, shard.name, shard.object_count, shard.vector_indexing_status, shard.loaded])
# print(shard_table)

In [ ]:
from collections import defaultdict
import pandas as pd

def check_shard_consistency(node_info):
    """
    Check consistency of shard object counts across nodes.

    :param node_info: Output of client.cluster.nodes(output="verbose").
    """
    # Group shards by collection and shard name
    shard_data = defaultdict(list)
    for node in node_info:
        for shard in node.shards:  # Access shards as attributes
            shard_key = (shard.collection, shard.name)
            shard_data[shard_key].append((node.name, shard.object_count))

    # Check for inconsistencies
    inconsistent_shards = []
    for (collection, shard_name), details in shard_data.items():
        object_counts = [obj_count for _, obj_count in details]
        if len(set(object_counts)) > 1:  # Inconsistent if counts are not identical
            for node_name, object_count in details:
                inconsistent_shards.append({
                    "Collection": collection,
                    "Shard": shard_name,
                    "Node": node_name,
                    "Object Count": object_count,
                })

    # Display results
    if inconsistent_shards:
        df_inconsistent_shards = pd.DataFrame(inconsistent_shards)
        print("Inconsistent Shards Found:")
        print(df_inconsistent_shards.to_string(index=False))
    else:
        print("All shards are consistent.")

# Example usage:

# Fetch node information dynamically from your cluster
node_info = client.cluster.nodes(output="verbose")
# Run the function with your node_info
check_shard_consistency(node_info)